# Mood-aware book recommender system. Data Cleaning. 

In [1]:
# Import the necessary libraries to perform data cleaning
import os
import math
import re
from collections import Counter
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
import certifi
import numpy as np
import matplotlib.pyplot as plt
from langdetect import detect, LangDetectException
import seaborn as sns
from html import unescape

We now have to import the dataset from MongoDB

In [2]:
# We have to import the dataset from MongoDB. 
# Load in the credentials
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

# Connect to the MongoDB database
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
db = client["BookProject"]
collection = db["books"]
# Convert the collection to a Pandas DataFrame
# We use list() to fetch all documents and pd.DataFrame() to structure them
df = pd.DataFrame(list(collection.find()))

Before we clean the dataset, we want to create a copy, which we will work on. Once the data cleaning process is finalised, we will save the data into a new dataset. 

In [3]:
# Let's create a copy of the original dataframe to work with
df_clean = df.copy()


In [4]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   _id               10000 non-null  object 
 1   book_id           10000 non-null  str    
 2   title             10000 non-null  str    
 3   description       10000 non-null  str    
 4   average_rating    10000 non-null  float64
 5   num_pages         10000 non-null  int64  
 6   ratings_count     10000 non-null  int64  
 7   publication_year  10000 non-null  str    
 8   url               10000 non-null  str    
 9   image_url         10000 non-null  str    
 10  authors           10000 non-null  object 
 11  tags              10000 non-null  object 
dtypes: float64(1), int64(2), object(3), str(6)
memory usage: 937.6+ KB


# Filtering by language
Before we do any cleaning, let's filter by language since we only want to keep books with English descriptions

In [5]:
def detect_language(text):
    """Detect language, return 'unknown' if detection fails"""
    try:
        if len(text.strip()) < 20:  # Too short to detect reliably
            return 'unknown'
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Detect language
print("Detecting languages... (this may take a minute)")
df_clean['detected_language'] = df_clean['description'].apply(detect_language)


Detecting languages... (this may take a minute)


In [6]:

# Check distribution
print("\nLanguage distribution:")
print(df_clean['detected_language'].value_counts().head(10))



Language distribution:
detected_language
en         6587
unknown    1769
es          247
it          209
cy          158
id          137
fr          132
pt          123
de          120
nl           71
Name: count, dtype: int64


In [7]:

# Filter for English only
df_clean = df_clean[df_clean['detected_language'] == 'en']
print(f"\nAfter English filter: {len(df_clean)} books")


After English filter: 6587 books


## Handling missing data
We want to 
- Drop entries with missing publication years
- Drop rows with empty descriptions
- Drop rows with empty tags list 
- Drop rows with empty authors list

In [8]:
# Remove empty descriptions
df_clean = df_clean[df_clean['description'].str.strip() != '']

# Remove empty tags
df_clean = df_clean[df_clean['tags'].apply(lambda x: len(x) > 0)]

# Remove empty authors
df_clean = df_clean[df_clean['authors'].apply(lambda x: len(x) > 0)]

print(f"After removing empty core fields: {len(df_clean)} books")

After removing empty core fields: 6521 books


## Invalid or Suspicious data
- Drop books with publication year before 1450 or after 2026 (the current year)
- Drop books that have duplicate titles (keeping the first instance of duplicate)
- Drop rows with descriptions shorter than 20 characters (arbitrary length chosen as a measure of a short description which won't be useful for the recommendation system)
- We will keep books with duplicate descriptions

Removing the duplicates:

In [9]:
# Remove duplicate titles (keep highest rated)
df_clean = df_clean.sort_values(by=['title', 'average_rating'], ascending=[True, False])
df_clean = df_clean.drop_duplicates(subset=['title'], keep='first')

# Remove duplicate descriptions (keep first occurrence)
df_clean = df_clean.drop_duplicates(subset=['description'], keep='first')

print(f"After removing duplicates: {len(df_clean)} books")

After removing duplicates: 6433 books


Invalid publication years (before printing):

In [10]:
# Convert to numeric and filter
current_year = pd.Timestamp.now().year
df_clean['publication_year'] = pd.to_numeric(df_clean['publication_year'], errors='coerce')
df_clean = df_clean.dropna(subset=['publication_year'])
df_clean = df_clean[(df_clean['publication_year'] >= 1450) & (df_clean['publication_year'] <= current_year)]

print(f"After publication year filter: {len(df_clean)} books")

After publication year filter: 5113 books


Remove short descriptions:

In [11]:
# Remove descriptions shorter than 50 characters
df_clean = df_clean[df_clean['description'].str.len() >= 50]

print(f"After removing short descriptions: {len(df_clean)} books")

After removing short descriptions: 5089 books


Clean html from the descriptions

In [12]:
def clean_html_and_whitespace(text):
    """Remove HTML tags/entities, normalize whitespace"""
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Unescape HTML entities
    text = unescape(text)
    # Normalize whitespace: remove tabs, newlines, multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

df_clean['description_cleaned'] = df_clean['description'].apply(clean_html_and_whitespace)

# Verify
print("Sample cleaned description:")
print(df_clean['description_cleaned'].iloc[0][:200])

Sample cleaned description:
It's the early 1980s, MTV is in its infancy, the Internet does not exist, Ronald Reagan is president and yuppies are ruling Wall Street. Edie is a naive NYU student desperate to lose her virginity and


## Filter and Clean tags
We want to filter out non-content shelves from tags:
- Format tags: to-read, currently-reading, owned, kindle, ebook, audiobook, library, borrowed, tbr, wishlist, default, etc
- Temporal tags: read-in-2016, read-2015, etc
- Rating tags: 4-stars, 5-stars, etc
- Personal organisation: my-library, have, reviewed, finished, dnf, did-not-finish, etc. 
- Reading platform: audible, nook, calibre, arc, ibooks, etc. 
- Physical format: paperback, hardcover, home-library, etc. 


We want to keep genre/mood shelves, which is the information based on which we can recommend books:
- Genres: fiction, fantasy, romance, thriller, sci-fi, etc
- Moods: adventure, emotional, dark, cozy, funny, etc. 

We will have to normalise/standardise variations such as (scifi-> science-fiction, etc).

In [13]:
all_tags = []
for tags in df['tags']:
    all_tags.extend(tags)
print(f"Number of unique tags before cleaning: {len(set(all_tags))}")

Number of unique tags before cleaning: 115845


In [14]:
# All logistical/organizational shelves to remove
LOGISTICAL_SHELVES = {
    # Reading status
    'to-read', 'currently-reading', 'read', 'dnf', 'did-not-finish', 
    'abandoned', 'unfinished', 'finished', 'unread', 're-read', 'reread',
    'didn-t-finish',
    
    # Ownership/format
    'owned', 'own', 'own-it', 'i-own', 'have', 'books-i-own', 'owned-books',
    'kindle', 'ebook', 'e-book', 'ebooks', 'e-books', 'audiobook', 'audiobooks',
    'audio', 'audio-book', 'audio-books', 'audible', 'ibooks',
    'paperback', 'hardcover', 'library', 'library-books', 'borrowed', 'to-read-own',
    
    # Organization/tracking
    'default', 'favorites', 'favourites', 'favorite', 'my-books', 'books',
    'my-library', 'home-library', 'tbr', 'wish-list', 'to-buy', 'maybe',
    'reviewed', 'arc', 'calibre', 'nook', 'bookshelf', 'on-my-shelf', 
    'books-i-have', 'on-hold', 'owned-to-read', 'need-to-buy',
    
    # Additional filters
    'kindle-books', 'library-book', 'to-read-fiction', 'shelfari-favorites',
    'first-reads', 'novels', 'novel', 'book-club', 'english', 'school',
    'general-fiction', 'adult-fiction', 'kindle-unlimited',
    
    # Structural
    'on-kindle', 'want-to-read', 'own-to-read', 'netgalley', 'want',
    'stand-alone', 'part-of-a-series', 'series',
    
    # Geographic/temporal
    'american', 'british', 'england', '20th-century', 'europe',
    
    # Compound tags
    'sci-fi-fantasy', 'mystery-suspense',
    
    # Favorites/recommendations (subjective organization)
    'favorite-books', 'all-time-favorites', 'recommended', 'must-read',
}

# Patterns to filter
TEMPORAL_PATTERN = r'^read-?(in-)?20\d{2}$'  
TEMPORAL_PATTERN_REVERSE = r'^20\d{2}-reads?$'
RATING_PATTERN = r'^\d-stars?$'

# Normalization dictionary
TAG_NORMALIZATION = {
    'nonfiction': 'non-fiction',
    'sci-fi': 'science-fiction',
    'scifi': 'science-fiction',
    'ya': 'young-adult',
    'childrens': 'children',
    'children-s': 'children',
    'children-s-books': 'children',
    'childrens-books': 'children',
    'kids': 'children',
    'classic': 'classics',
    'contemporary-romance': 'romance',
    'contemporary-fiction': 'contemporary',
    'mystery-thriller': 'thriller',
    'historical': 'historical-fiction',
    'mysteries': 'mystery',
    'thrillers': 'thriller',
    'teen': 'young-adult',
    'literary': 'literary-fiction',
    'funny': 'humor',
    'fantasy-sci-fi': 'fantasy',
}

In [15]:
def clean_tags(tags_list):
    """Remove logistical shelves, normalize variations, keep only content shelves."""
    cleaned = []
    
    for tag in tags_list:
        tag_lower = tag.lower().strip()
        
        # Remove punctuation (keep hyphens for compound tags like 'science-fiction')
        tag_lower = re.sub(r'[^\w\s-]', '', tag_lower)
        
        # Skip if in logistical shelves
        if tag_lower in LOGISTICAL_SHELVES:
            continue
            
        # Skip if tag is just a number
        if tag_lower.isdigit():
            continue
            
        # Skip temporal/rating patterns
        if re.match(TEMPORAL_PATTERN, tag_lower):
            continue
        if re.match(TEMPORAL_PATTERN_REVERSE, tag_lower):
            continue
        if re.match(RATING_PATTERN, tag_lower):
            continue
            
        # Normalize variations
        tag_normalized = TAG_NORMALIZATION.get(tag_lower, tag_lower)
        
        # Add to cleaned list (avoid duplicates within same book)
        if tag_normalized not in cleaned:
            cleaned.append(tag_normalized)
    
    return cleaned

df_clean['tags_cleaned'] = df_clean['tags'].apply(clean_tags)

# Remove books with no tags left after cleaning
df_clean = df_clean[df_clean['tags_cleaned'].apply(len) > 0]

print(f"After tag cleaning: {len(df_clean)} books")

After tag cleaning: 4976 books


### Validating numeric features


In [16]:
# Ensure average_rating is between 0 and 5
df_clean = df_clean[(df_clean['average_rating'] >= 0) & (df_clean['average_rating'] <= 5)]

# Ensure ratings_count >= 0
df_clean = df_clean[df_clean['ratings_count'] >= 0]

# Ensure num_pages >= 0 (0 is ok, means unknown)
df_clean = df_clean[df_clean['num_pages'] >= 0]

print(f"After numeric validation: {len(df_clean)} books")

After numeric validation: 4976 books


In [17]:
# Description length (characters)
df_clean['description_length'] = df_clean['description_cleaned'].str.len()

# Description word count
df_clean['description_word_count'] = df_clean['description_cleaned'].apply(lambda x: len(x.split()))

# Number of tags after cleaning
df_clean['num_tags'] = df_clean['tags_cleaned'].apply(len)

print("\nDerived features summary:")
print(f"Avg description length: {df_clean['description_length'].mean():.0f} chars")
print(f"Avg description word count: {df_clean['description_word_count'].mean():.0f} words")
print(f"Avg tags per book: {df_clean['num_tags'].mean():.1f}")


Derived features summary:
Avg description length: 867 chars
Avg description word count: 144 words
Avg tags per book: 46.0


In [18]:
# Keep only needed columns
final_columns = [
    'book_id',
    'title',
    'description_cleaned',
    'description_length',
    'description_word_count',
    'average_rating',
    'num_pages',
    'ratings_count',
    'publication_year',
    'authors',
    'tags_cleaned',
    'num_tags'
]

df_final = df_clean[final_columns].copy()

# Final validation
print("\n=== FINAL DATASET VALIDATION ===")
print(f"Original dataset: 10,000 books")
print(f"Final cleaned dataset: {len(df_final)} books")
print(f"Rows removed: {10000 - len(df_final)} ({((10000 - len(df_final)) / 10000) * 100:.1f}%)")
print(f"\nMissing values check:")
print(df_final.isnull().sum())
print(f"\nNumeric ranges:")
print(f"  Ratings: {df_final['average_rating'].min():.2f} - {df_final['average_rating'].max():.2f}")
print(f"  Pages: {df_final['num_pages'].min()} - {df_final['num_pages'].max()}")
print(f"  Year: {int(df_final['publication_year'].min())} - {int(df_final['publication_year'].max())}")

df_final.head()


=== FINAL DATASET VALIDATION ===
Original dataset: 10,000 books
Final cleaned dataset: 4976 books
Rows removed: 5024 (50.2%)

Missing values check:
book_id                   0
title                     0
description_cleaned       0
description_length        0
description_word_count    0
average_rating            0
num_pages                 0
ratings_count             0
publication_year          0
authors                   0
tags_cleaned              0
num_tags                  0
dtype: int64

Numeric ranges:
  Ratings: 0.00 - 5.00
  Pages: 0 - 2969
  Year: 1839 - 2019


,book_id,title,description_cleaned,description_length,description_word_count,average_rating,num_pages,ratings_count,publication_year,authors,tags_cleaned,num_tags
5119,25341926,"""Love, Loss and Longing in the Age of Reagan: ...","It's the early 1980s, MTV is in its infancy, t...",699,122,3.39,241,18,2016.0,[2020637],"[new-adult, started-reading, womens-fiction, 2...",15
6345,35040603,#Triggerwarning,When a college dean's affair with a student go...,478,70,4.00,250,5,2017.0,[14410739],"[giveaways-entered, goodread-wins, won-firstre...",5
705,10318640,'I am an American': Filming the Fear of Differ...,From Samuel Huntington's highly controversial ...,1365,212,3.83,224,6,2011.0,[67970],"[world-interests, border]",2
1754,18665182,'Tis the Season to Get Lucky,Unlucky-in-love PR manager Kate Braithwaite do...,503,83,3.77,70,79,2013.0,[786456],"[christmas, romance, holiday, contemporary, no...",64
1787,18665181,'Tis the Season to be Kissed,A down-on-her-romantic-luck kindergarten teach...,239,39,3.83,70,148,2013.0,[1128457],"[romance, contemporary, holiday, novella, mili...",79


In [22]:
output_path = '../data/processed/books_cleaned.csv'
df_final.to_csv(output_path, index=False)

print(f"\n✓ Cleaned dataset saved to: {output_path}")
print(f"✓ Ready for feature engineering")


✓ Cleaned dataset saved to: ../data/processed/books_cleaned.csv
✓ Ready for feature engineering
